## 96. Global Coverage

### 1. Packages

In [282]:
# Packages
import geopandas as gpd
import glob
import os
import numpy as np
import rioxarray as rxr
from tqdm import tqdm
from pyproj import Geod
import pandas as pd
from google.cloud import storage

# TODO: align the computations with the tables in the SQO! 
# TODO: Horizontal and vertical coverage; area stats to be normalized w.r.t. IPCC ref region area to compare? 
# TODO: visualize hor & vertical coverage per IPCC region?!

### 2. Settings

In [104]:
# Settings
mode = 'intertidal_improved_100m_global'   # Specify mode, either 'intertidal' or 'subtidal'
start_date = '2021-01-01'                  # Start date of the composites
stop_date = '2022-01-01'                   # End date of the composites
compo_int = 12                             # Composite interval [months]
compo_len = 12                             # Composite length [months]
scale = 100                                # Output resolution of the image [m]
crs = 'EPSG:4326'                          # Output projection of the image

upscale = 100                              # Upscaling factor for the image
res_arc_min = 1/16                         # resolution of the image in arc minutes
res_deg = res_arc_min / 60                 # resolution of the image in degrees


# Tiling (see https://www.openearth.nl/rws-bathymetry/2019.html)
zoom_levels = [9, 10, 11] # list with zoom levels
zoom_level = zoom_levels[1] # zoom level to be used

# Directories and files
dir_path_base = r'p:\11209821-cmems-global-sdb'
file_path_tiles = os.path.join(dir_path_base, '00_miscellaneous', 'AOI_polygons_world', f'df_boxes_world_Z{zoom_level}_filtered_v2.parquet') # Tiles file
file_path_sdbs = glob.glob(os.path.join(dir_path_base, '01_intertidal', '02_data', '05_calibrated', mode, '05_reprojected', '*.tif'))        # SDB files
file_path_sdbs = [file_path for file_path in file_path_sdbs if not file_path.endswith('.tif.aux.xml')]
file_path_progress = os.path.join(dir_path_base, '01_intertidal', '02_data', '05_calibrated', 'progress_global')
print('Number of SDB files:', len(file_path_sdbs))

Number of SDB files: 11893


### 3. Read tiles

In [114]:
# Read tiles
gdf_tiles_org = gpd.read_parquet(file_path_tiles)
gdf_tiles_org['nearest_station_distance'] = gdf_tiles_org['nearest_station_distance']/1000  # Convert to km
gdf_tiles_org['intertidal_coverage'] = gdf_tiles_org['intertidal_coverage']*100  # Convert to percentage
gdf_tiles_org['intertidal_coverage_ed'] = gdf_tiles_org['intertidal_coverage_ed']*100  # Convert to percentage
gdf_tiles = gdf_tiles_org.copy()
gdf_tiles['processed'] = (gdf_tiles['nearest_station_distance'] <= 37) & (gdf_tiles['intertidal_coverage'] >= 1) & (gdf_tiles['intertidal_coverage_ed'] > 0)
gdf_tiles = gdf_tiles[gdf_tiles['processed']]

# Add sdb file_path to tiles
for i, row in tqdm(gdf_tiles.iterrows(), total=gdf_tiles.shape[0]):
    file_path = [file_path for file_path in file_path_sdbs if os.path.basename(file_path).startswith(row['name'])]
    gdf_tiles.at[i, 'file_path'] = file_path[0] if file_path else None

gdf_tiles.head()

100%|██████████| 18797/18797 [39:14<00:00,  7.98it/s]


,name,id,tx,ty,zoom,geometry,path,ref_region,area_perc_org,area_perc_red,nearest_station_id,nearest_station_longitude,nearest_station_latitude,nearest_station_distance,intertidal_coverage,intertidal_coverage_ed,processed,file_path
0,z10_x561_y116,2195,561.0,116.0,10,"POLYGON ((17.22656 79.87430, 17.57813 79.87430...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,2.32,1.18,station 07499,17.33421,79.92993,3.062257,8.286716,1.182026,True,NaN
1,z10_x667_y108,13635,667.0,108.0,10,"POLYGON ((54.49219 80.35700, 54.84375 80.35700...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,1.29,0.14,station 07502,54.76023,80.27526,12.472398,16.578475,0.135397,True,NaN
2,z10_x566_y141,2760,566.0,141.0,10,"POLYGON ((18.98438 78.20656, 19.33594 78.20656...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,0.71,0.07,station 07487,18.46672,77.95945,35.249050,2.918707,0.157026,True,NaN
3,z10_x545_y140,491,545.0,140.0,10,"POLYGON ((11.60156 78.27820, 11.95313 78.27820...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,22.14,21.76,station 07438,11.91355,78.40147,10.215705,22.184213,21.787025,True,NaN
7,z10_x550_y146,1037,550.0,146.0,10,"POLYGON ((13.35938 77.84185, 13.71094 77.84185...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,17.38,16.74,station 07491,13.86779,78.09662,25.418648,17.374234,16.777855,True,NaN


In [230]:
# Remove tiles without SDB file_path
gdf_tiles_red = gdf_tiles[gdf_tiles['file_path'].notna()]
gdf_tiles_red = gdf_tiles_red.drop_duplicates(subset='file_path', keep="last") # remvoe duplicates z10_x384_y51 & z10_x384_y515 (this one is good)

gdf_tiles_red.shape

# check duplicates
# for i, j in zip(gdf_tiles_red.name, gdf_tiles_red['file_path']):
#     if i == "z10_x397_y519":
#         print(i, j)
# len(np.unique(gdf_tiles_red.file_path))
# duplicates = gdf_tiles_red[gdf_tiles_red.duplicated(subset=['file_path'], keep='first')]
# for i, j in zip(duplicates.name, duplicates.file_path):
#     print(i, j)

(11893, 21)

### 4. Check cloud progress

In [177]:
# open excel file
prog_excel = pd.read_excel(r"p:\11209821-cmems-global-sdb\01_intertidal\02_data\05_calibrated\progress_global\global_progress.xlsx", header=1)

# find index where column number of tiles has a number
last_complete_idx = min(prog_excel["number_of_tiles"][~prog_excel["number_of_tiles"].notna()].index)
ref_regions_comp = list(prog_excel["ref_region"].iloc[0:last_complete_idx])

# tiles combined submitted for processing
tiles_submitted = gdf_tiles[gdf_tiles["ref_region"].isin(ref_regions_comp)]
print("Submitted:",len(tiles_submitted)*2)

Submitted: 34630


In [178]:
# stored in the cloud

# cloud variables
source_project = "cmems-sdb-11209821-002"
source_bucket_name = "cmems-isdb"
source_bucket_proj_tiff = "intertidal_improved_100m_global/z10"
source_bucket_proj_meta = "intertidal_improved_100m_global_meta/z10"

# create python storage clients for project
source_client = storage.Client(project=source_project)
source_bucket = source_client.bucket(source_bucket_name)

# List all files under the source prefix
blobs_data = list(source_client.list_blobs(source_bucket, prefix=source_bucket_proj_tiff))
blobs_meta = list(source_client.list_blobs(source_bucket, prefix=source_bucket_proj_meta))
blobs_data_name = [blobs.name[len(source_bucket_proj_tiff)-3:].strip(".tif") for blobs in blobs_data]
blobs_meta_name = [blobs.name[len(source_bucket_proj_meta)-3:].strip(".csv") for blobs in blobs_meta]
print("tiffs:", len(blobs_data_name))
print("meta:", len(blobs_meta_name))

# combined processed (tiffs + csvs)
blobs_proc = blobs_data_name + blobs_meta_name
print("total:", len(blobs_proc))

# 

tiffs: 15847
meta: 16495
total: 32342


In [179]:
# combined successes (tiffs and csvs)
comb_succ = [blobs for blobs in blobs_data_name if blobs in blobs_meta_name]
print("combined succes:", len(comb_succ))

# succes for tiffs and failures for csvs
succ_tiff_fail_csv = [blobs for blobs in blobs_data_name if blobs not in blobs_meta_name]
print("failed csv:",len(succ_tiff_fail_csv))

# succes for csvs and failures for tiffs
succ_csv_fail_tiff = [blobs for blobs in blobs_meta_name if blobs not in blobs_data_name]
print("failed tiff:", len(succ_csv_fail_tiff))

# failures for both tiffs and csvs
blobs_proc_name = [blob.split("/t")[0].replace("/", "_") for blob in blobs_proc]
comb_fail = tiles_submitted["name"][~tiles_submitted["name"].isin(blobs_proc_name)]
print("combined failed:", len(comb_fail), "*2=", len(comb_fail)*2)

# failed in total
print("failed total:", len(tiles_submitted)*2-len(blobs_proc))

combined succes: 15847
failed csv: 0
failed tiff: 648
combined failed: 820 *2= 1640
failed total: 2288


In [180]:
# for i in succ_csv_fail_tiff_name:
#     if "z10_x286_y97" in i:
#         print(i)

In [181]:
# build a dataframe of failed tasks (using three options above)
comb_succ_name = [blob.split("/t")[0].replace("/", "_") for blob in comb_succ]
succ_tiff_fail_csv_name = [blob.split("/t")[0].replace("/", "_") for blob in succ_tiff_fail_csv]
succ_csv_fail_tiff_name = [blob.split("/t")[0].replace("/", "_") for blob in succ_csv_fail_tiff]
comb_fail_name = [blob.split("/t")[0].replace("/", "_") for blob in comb_fail]
failed_list = succ_tiff_fail_csv_name + succ_csv_fail_tiff_name + comb_fail_name
print("failed total unique:", len(failed_list))

# make df
failed_df = tiles_submitted[tiles_submitted["name"].isin(failed_list)]

#print(failed_df.shape)
print(failed_df["ref_region"].value_counts())

failed total unique: 1468
ref_region
GIC     714
RAR     182
NEN     114
NWN      85
SEA      60
ARP      41
MED      28
EAS      19
SAS      18
NEU      18
SAH      18
RFE      17
NCA      14
ENA      13
EPO      11
CAU      11
SSA       9
WCA       9
SOO       9
MDG       7
CAR       6
NAU       5
NES       5
NZ        5
SPO       4
SEAF      4
SCA       4
WCE       4
WSAF      4
ESAF      4
SES       3
WAF       3
CAF       3
SAU       2
BOB       2
NPO       2
NEAF      2
NAO       2
WNA       2
NWS       1
EIO       1
SWS       1
EEU       1
NSA       1
Name: count, dtype: int64


In [182]:
# check the tiles not in GIC & RAR

# GDF tiles with GIC & RAR as ref_region
gdf_tiles_ad = gdf_tiles[gdf_tiles['ref_region'].isin(['GIC', 'RAR'])]
gdf_tiles_rd = gdf_tiles[~gdf_tiles['ref_region'].isin(['GIC', 'RAR'])]

# check how many files are in the data before running RAR & GIC
comb_succ_name_rd = [name for name in comb_succ_name if name in gdf_tiles_rd["name"].to_list()]
succ_tiff_fail_csv_name_rd = [name for name in succ_tiff_fail_csv_name if name in gdf_tiles_rd["name"].to_list()]
succ_csv_fail_tiff_name_rd = [name for name in succ_csv_fail_tiff_name if name in gdf_tiles_rd["name"].to_list()]
comb_fail_name_rd = [name for name in comb_fail_name if name in gdf_tiles_rd["name"].to_list()]
print(len(comb_succ_name_rd), len(succ_tiff_fail_csv_name_rd), len(succ_csv_fail_tiff_name_rd), len(comb_fail_name_rd))

# stats on complete and failed for an ref area (like GIC or RAR), test for NEN
ref_reg = "GIC"
gdf_tiles_stats = gdf_tiles[gdf_tiles['ref_region'].isin([ref_reg])]
succ_tiff_fail_csv_name_stats = [name for name in succ_tiff_fail_csv_name if name in gdf_tiles_stats["name"].to_list()]
succ_csv_fail_tiff_name_stats = [name for name in succ_csv_fail_tiff_name if name in gdf_tiles_stats["name"].to_list()]
comb_fail_name_stats = [name for name in comb_fail_name if name in gdf_tiles_stats["name"].to_list()]
comb_succ_name_stats = [name for name in comb_succ_name if name in gdf_tiles_stats["name"].to_list()]
print(len(comb_succ_name_stats), len(succ_tiff_fail_csv_name_stats), len(succ_csv_fail_tiff_name_stats), len(comb_fail_name_stats))
print("completed:", len(comb_succ_name_stats)*2+ len(succ_tiff_fail_csv_name_stats)+len(succ_csv_fail_tiff_name_stats))
print ("total failed:", len(comb_fail_name_stats)*2+len(succ_tiff_fail_csv_name_stats)+len(succ_csv_fail_tiff_name_stats))

12998 0 239 333
1285 0 285 429
completed: 2855
total failed: 1143


In [183]:
# save as parquet   
#failed_df.to_parquet(os.path.join(file_path_progress, f'failed_tiles_default_run_no_RAR_and_GIC.parquet'), index=False)

### 5. Calculate horizontal area based on feasibility map

In [231]:
# Reproject tiles to metres
gdf_tiles = gdf_tiles.to_crs('EPSG:6933') # Project to equal-area for grid-based metrics, EPSG 3857 is not to be used for area or dist (only mapping)

# Calculate area of the tiles
gdf_tiles['tile_area'] = gdf_tiles['geometry'].area / 1e6  # Convert to km2

# Calculate feasibility map coverage
gdf_tiles['intertidal_area'] = gdf_tiles['tile_area'] * gdf_tiles['intertidal_coverage']/100 # km2
gdf_tiles['intertidal_area_ed'] = gdf_tiles['tile_area'] * gdf_tiles['intertidal_coverage_ed']/100 # km2

# do the same for the gdf_tiles_red
gdf_tiles_red = gdf_tiles_red.to_crs('EPSG:6933') # Project to equal-area for grid-based metrics, EPSG 3857 is not to be used for area or dist (only mapping)
gdf_tiles_red['tile_area'] = gdf_tiles_red['geometry'].area / 1e6  # Convert to km2
gdf_tiles_red['intertidal_area'] = gdf_tiles_red['tile_area'] * gdf_tiles_red['intertidal_coverage']/100 # km2
gdf_tiles_red['intertidal_area_ed'] = gdf_tiles_red['tile_area'] * gdf_tiles_red['intertidal_coverage_ed']/100 # km2

# # most accurate but need loop
# gdf_tiles = gdf_tiles.to_crs('EPSG:4326')
# # WGS84 ellipsoid
# geod = Geod(ellps="WGS84")
# # Compute area (in square meters), always positive
# area, _ = geod.geometry_area_perimeter(gdf_tiles.iloc[0].geometry)
# area_geod = abs(area)
# #print(area/1e6)

In [233]:
gdf_tiles.head()

,name,id,tx,ty,zoom,geometry,path,ref_region,area_perc_org,area_perc_red,...,nearest_station_longitude,nearest_station_latitude,nearest_station_distance,intertidal_coverage,intertidal_coverage_ed,processed,file_path,tile_area,intertidal_area_ed,intertidal_area
0,z10_x561_y116,2195,561.0,116.0,10,"POLYGON ((1662126.937 7226866.115, 1696047.895...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,2.32,1.18,...,17.33421,79.92993,3.062257,8.286716,1.182026,True,NaN,47.352511,0.559719,3.923968
1,z10_x667_y108,13635,667.0,108.0,10,"POLYGON ((5257748.475 7237575.520, 5291669.433...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,1.29,0.14,...,54.76023,80.27526,12.472398,16.578475,0.135397,True,NaN,42.988448,0.058205,7.126829
2,z10_x566_y141,2760,566.0,141.0,10,"POLYGON ((1831731.727 7185891.778, 1865652.685...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,0.71,0.07,...,18.46672,77.95945,35.249050,2.918707,0.157026,True,NaN,63.989524,0.100480,1.867667
3,z10_x545_y140,491,545.0,140.0,10,"POLYGON ((1119391.611 7187778.209, 1153312.569...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,22.14,21.76,...,11.91355,78.40147,10.215705,22.184213,21.787025,True,NaN,63.225655,13.774989,14.026114
7,z10_x550_y146,1037,550.0,146.0,10,"POLYGON ((1288996.400 7176112.427, 1322917.358...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,17.38,16.74,...,13.86779,78.09662,25.418648,17.374234,16.777855,True,NaN,67.946233,11.399921,11.805137


### 6. Calculate horizontal area and vert. depth based on sdb results (after post-processing, i.e. applying masks)

In [285]:
# Reproject tiles to metres
gdf_tiles_red = gdf_tiles_red.to_crs('EPSG:4326') # back to normal
#gdf_tiles_red_m = gdf_tiles_red.to_crs('EPSG:6933') # Project to equal-area for grid-based metrics, EPSG 3857 is not to be used for area or dist (only mapping)

# # Get area of the sdb files
for i, row in tqdm(gdf_tiles_red.iterrows(), total=gdf_tiles_red.shape[0]):
    # Read sdb file
    
    ds = rxr.open_rasterio(row['file_path'])
    
    # Clip to tile
    ds = ds.rio.clip([row['geometry']])

    #print(ds[0,:,:], ds[0,:,:].size)

    # Get horizontal coverage of the tile (number of non-null pixels)
    intertidal_coverage_sdb = ds.notnull().sum() / ds.size * 100 # non null pixels / total pixels * 100
    gdf_tiles_red.at[i, 'intertidal_coverage_sdb'] = intertidal_coverage_sdb
    gdf_tiles_red.at[i, 'intertidal_area_sdb'] = gdf_tiles_red.at[i, 'tile_area'] * intertidal_coverage_sdb/100
    gdf_tiles_red.at[i, 'intertidal_coverage_sdb_feas'] = gdf_tiles_red.at[i, 'intertidal_area_sdb'] / gdf_tiles_red.at[i, 'intertidal_area'] * 100 
    #gdf_tiles_red.at[i, 'intertidal_area_sdb_feas'] = (gdf_tiles_red.at[i, 'intertidal_coverage_sdb_feas']/100) * gdf_tiles_red.at[i, 'intertidal_area'] 
    gdf_tiles_red.at[i, 'intertidal_coverage_sdb_feas_ed'] = gdf_tiles_red.at[i, 'intertidal_area_sdb'] / gdf_tiles_red.at[i, 'intertidal_area_ed'] * 100 
    #gdf_tiles_red.at[i, 'intertidal_area_sdb_feas_ed'] = (gdf_tiles_red.at[i, 'intertidal_coverage_sdb_feas_ed']/100) * gdf_tiles_red.at[i, 'intertidal_area_ed']

    # Get vertical coverage of the tile
    max_depth = np.nanmax(ds[0,:,:].values)
    min_depth = np.nanmin(ds[0,:,:].values)
    gdf_tiles_red.at[i, "max_depth"] = max_depth
    gdf_tiles_red.at[i, "min_depth"] = min_depth
    gdf_tiles_red.at[i, "depth_range"] = max_depth - min_depth


  0%|          | 0/11893 [00:00<?, ?it/s]

  2%|▏         | 217/11893 [00:38<36:02,  5.40it/s]C:\Users\kras\AppData\Local\Temp\ipykernel_5252\4231832875.py:26: RuntimeWarning: All-NaN slice encountered
  max_depth = np.nanmax(ds[0,:,:].values)
C:\Users\kras\AppData\Local\Temp\ipykernel_5252\4231832875.py:27: RuntimeWarning: All-NaN slice encountered
  min_depth = np.nanmin(ds[0,:,:].values)
  3%|▎         | 366/11893 [01:06<35:48,  5.36it/s]C:\Users\kras\AppData\Local\Temp\ipykernel_5252\4231832875.py:26: RuntimeWarning: All-NaN slice encountered
  max_depth = np.nanmax(ds[0,:,:].values)
C:\Users\kras\AppData\Local\Temp\ipykernel_5252\4231832875.py:27: RuntimeWarning: All-NaN slice encountered
  min_depth = np.nanmin(ds[0,:,:].values)
  6%|▌         | 655/11893 [01:54<30:02,  6.24it/s]C:\Users\kras\AppData\Local\Temp\ipykernel_5252\4231832875.py:26: RuntimeWarning: All-NaN slice encountered
  max_depth = np.nanmax(ds[0,:,:].values)
C:\Users\kras\AppData\Local\Temp\ipykernel_5252\4231832875.py:27: RuntimeWarning: All-NaN slice e

### 7. Sum horizontal global coverage per region

#### for all tiles

In [287]:
# for all tiles

# Reproject tiles to metres
gdf_tiles_org = gdf_tiles_org.to_crs('EPSG:6933') # Project to equal-area for grid-based metrics, EPSG 3857 is not to be used for area or dist (only mapping)

# Calculate area of the tiles
gdf_tiles_org['tile_area'] = gdf_tiles_org['geometry'].area / 1e6  # Convert to km2

# Calculate coverage
gdf_tiles_org['intertidal_area_ed'] = gdf_tiles_org['tile_area'] * gdf_tiles_org['intertidal_coverage_ed']/100

In [288]:
gdf_tiles_org.head()

,name,id,tx,ty,zoom,geometry,path,ref_region,area_perc_org,area_perc_red,nearest_station_id,nearest_station_longitude,nearest_station_latitude,nearest_station_distance,intertidal_coverage,intertidal_coverage_ed,tile_area,intertidal_area_ed
0,z10_x561_y116,2195,561.0,116.0,10,"POLYGON ((1662126.937 7226866.115, 1696047.895...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,2.32,1.18,station 07499,17.33421,79.92993,3.062257,8.286716,1.182026,47.352511,0.559719
1,z10_x667_y108,13635,667.0,108.0,10,"POLYGON ((5257748.475 7237575.520, 5291669.433...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,1.29,0.14,station 07502,54.76023,80.27526,12.472398,16.578475,0.135397,42.988448,0.058205
2,z10_x566_y141,2760,566.0,141.0,10,"POLYGON ((1831731.727 7185891.778, 1865652.685...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,0.71,0.07,station 07487,18.46672,77.95945,35.249050,2.918707,0.157026,63.989524,0.100480
3,z10_x545_y140,491,545.0,140.0,10,"POLYGON ((1119391.611 7187778.209, 1153312.569...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,22.14,21.76,station 07438,11.91355,78.40147,10.215705,22.184213,21.787025,63.225655,13.774989
4,z10_x644_y109,11152,644.0,109.0,10,"POLYGON ((4477566.443 7236292.788, 4511487.401...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,0.42,0.00,station 07509,46.96189,80.35473,7.708247,1.606905,0.000000,43.511501,0.000000


In [289]:
# Group gdf by ref_region (FOR ALL TILES; 38639 along global coastline)

#gdf_tiles_grouped = gdf_tiles[['ref_region', 'tile_area', 'intertidal_area_ed', 'intertidal_area_sdb']].groupby('ref_region').agg(
#    {'tile_area': 'sum', 'intertidal_area_ed': 'sum', 'intertidal_area_sdb': 'mean'}).reset_index()
gdf_tiles_grouped_org = gdf_tiles_org[['ref_region', 'tile_area', 'intertidal_area_ed']].groupby('ref_region').agg(
    {'tile_area': 'sum', 'intertidal_area_ed': 'sum'}).reset_index()
gdf_tiles_grouped_org['tile_area'] = gdf_tiles_grouped_org['tile_area'].astype(int)
gdf_tiles_grouped_org['intertidal_area_ed'] = gdf_tiles_grouped_org['intertidal_area_ed'].astype(int)
#gdf_tiles_grouped_org['intertidal_area_sdb'] = gdf_tiles_grouped_org['intertidal_area_sdb'].astype(int)
#gdf_tiles_grouped_org['intertidal_area_sdb'] = gdf_tiles_grouped_org['intertidal_area_sdb'].round(2)

# Add total row
total_org = pd.DataFrame([{
    "ref_region": "TOTAL",
    "tile_area": gdf_tiles_grouped_org["tile_area"].sum(),
    "intertidal_area_ed": gdf_tiles_grouped_org["intertidal_area_ed"].sum()
}])

# Append the total row
grouped_with_total_org = pd.concat([gdf_tiles_grouped_org, total_org], ignore_index=True)

# Add percentage column
grouped_with_total_org["intertidal_area_ed_perc"] = (grouped_with_total_org["intertidal_area_ed"] / grouped_with_total_org["tile_area"]) * 100
grouped_with_total_org["intertidal_area_ed_perc"] = grouped_with_total_org["intertidal_area_ed_perc"].round(2)

grouped_with_total_org

# Do sorting on different columns to explain differences in the table (analyse!)

,ref_region,tile_area,intertidal_area_ed,intertidal_area_ed_perc
0,ARO,75856,1533,2.02
1,ARP,474295,14459,3.05
2,ARS,68251,602,0.88
3,BOB,66329,796,1.20
4,CAF,107177,1183,1.10
5,CAR,620288,20557,3.31
6,CAU,121182,3910,3.23
7,CNA,54382,5654,10.40
8,EAN,252529,5267,2.09
9,EAO,31898,252,0.79


#### for filtered tiles

In [290]:
# Group gdf by ref_region (FOR FILTERED TILES; 18797 along global coastline)
# gdf_tiles_grouped = gdf_tiles_red[['ref_region', 'tile_area', 'intertidal_area_ed', 'intertidal_area_sdb']].groupby('ref_region').agg(
#    {'tile_area': 'sum', 'intertidal_area_ed': 'sum', 'intertidal_area_sdb': 'sum'}).reset_index()
gdf_tiles_grouped = gdf_tiles[['ref_region', 'tile_area', 'intertidal_area_ed']].groupby('ref_region').agg(
    {'tile_area': 'sum', 'intertidal_area_ed': 'sum'}).reset_index()
gdf_tiles_grouped['tile_area'] = gdf_tiles_grouped['tile_area'].astype(int)
gdf_tiles_grouped['intertidal_area_ed'] = gdf_tiles_grouped['intertidal_area_ed'].astype(int)
#gdf_tiles_grouped['intertidal_area_sdb'] = gdf_tiles_grouped['intertidal_area_sdb'].astype(int)
#gdf_tiles_grouped['intertidal_area_sdb'] = gdf_tiles_grouped['intertidal_area_sdb'].round(2)

# Add total row
total_fil = pd.DataFrame([{
    "ref_region": "TOTAL",
    "tile_area": gdf_tiles_grouped["tile_area"].sum(),
    "intertidal_area_ed": gdf_tiles_grouped["intertidal_area_ed"].sum()
}])

# Append the total row
grouped_with_total_fil = pd.concat([gdf_tiles_grouped, total_fil], ignore_index=True)

# Add percentage column
grouped_with_total_fil["intertidal_area_ed_perc"] = (grouped_with_total_fil["intertidal_area_ed"] / grouped_with_total_fil["tile_area"]) * 100
grouped_with_total_fil["intertidal_area_ed_perc"] = grouped_with_total_fil["intertidal_area_ed_perc"].round(2)

grouped_with_total_fil

# Do sorting on different columns to explain differences in the table (analyse!)

,ref_region,tile_area,intertidal_area_ed,intertidal_area_ed_perc
0,ARO,37382,1282,3.43
1,ARP,275040,11669,4.24
2,ARS,5936,61,1.03
3,BOB,44176,677,1.53
4,CAF,68077,1074,1.58
5,CAR,330841,17702,5.35
6,CAU,61960,2279,3.68
7,CNA,46250,5643,12.20
8,EAN,53130,3445,6.48
9,EAO,10646,239,2.24


#### for post-processed tiles

In [297]:
# Group gdf by ref_region (FOR post-processed TILES; 11893 along global coastline)

gdf_tiles_grouped = gdf_tiles_red[['ref_region', 'tile_area', 'intertidal_area_ed', 'intertidal_area_sdb']].groupby('ref_region').agg(
   {'tile_area': 'sum', 'intertidal_area_ed': 'sum', 'intertidal_area_sdb': 'sum'}).reset_index()
# gdf_tiles_grouped = gdf_tiles[['ref_region', 'tile_area', 'intertidal_area_ed']].groupby('ref_region').agg(
#     {'tile_area': 'sum', 'intertidal_area_ed': 'sum'}).reset_index()
gdf_tiles_grouped['tile_area'] = gdf_tiles_grouped['tile_area'].astype(int)
gdf_tiles_grouped['intertidal_area_ed'] = gdf_tiles_grouped['intertidal_area_ed'].astype(int)
gdf_tiles_grouped['intertidal_area_sdb'] = gdf_tiles_grouped['intertidal_area_sdb'].astype(int)
#gdf_tiles_grouped['intertidal_area_sdb'] = gdf_tiles_grouped['intertidal_area_sdb'].round(2)

# Add total row
total = pd.DataFrame([{
    "ref_region": "TOTAL",
    "tile_area": gdf_tiles_grouped["tile_area"].sum(),
    "intertidal_area_ed": gdf_tiles_grouped["intertidal_area_ed"].sum(),
    "intertidal_area_sdb": gdf_tiles_grouped["intertidal_area_sdb"].sum()
}])

# Append the total row
grouped_with_total = pd.concat([gdf_tiles_grouped, total], ignore_index=True)

# Add percentage column
grouped_with_total["intertidal_area_ed_perc"] = (grouped_with_total["intertidal_area_ed"] / grouped_with_total["tile_area"]) * 100
grouped_with_total["intertidal_area_ed_perc"] = grouped_with_total["intertidal_area_ed_perc"].round(2)

grouped_with_total["intertidal_area_sdb_perc"] = (grouped_with_total["intertidal_area_sdb"] / grouped_with_total["tile_area"]) * 100
grouped_with_total["intertidal_area_sdb_perc"] = grouped_with_total["intertidal_area_sdb_perc"].round(2)

grouped_with_total["intertidal_area_sdb_ed_perc"] = (grouped_with_total["intertidal_area_sdb"] / grouped_with_total["intertidal_area_ed"]) * 100
grouped_with_total["intertidal_area_sdb_ed_perc"] = grouped_with_total["intertidal_area_sdb_ed_perc"].round(2)

grouped_with_total#.style.hide()

# Do sorting on different columns to explain differences in the table (analyse!)

,ref_region,tile_area,intertidal_area_ed,intertidal_area_sdb,intertidal_area_ed_perc,intertidal_area_sdb_perc,intertidal_area_sdb_ed_perc
0,ARP,193845,8358,1830,4.31,0.94,21.90
1,ARS,4488,60,2,1.34,0.04,3.33
2,BOB,20652,378,39,1.83,0.19,10.32
3,CAF,45395,889,171,1.96,0.38,19.24
4,CAR,232215,16800,813,7.23,0.35,4.84
5,CAU,44098,1943,329,4.41,0.75,16.93
6,CNA,45077,5542,1655,12.29,3.67,29.86
7,EAO,9125,239,70,2.62,0.77,29.29
8,EAS,459556,28473,8354,6.20,1.82,29.34
9,EAU,77893,2209,333,2.84,0.43,15.07


### Vertical coverage

In [368]:
# Group gdf by ref_region (FOR post-processed TILES; 11893 along global coastline)

gdf_tiles_grouped_vert = gdf_tiles_red[['ref_region', 'min_depth', 'max_depth', 'depth_range']].groupby('ref_region').agg(
   {'min_depth': 'min', 'max_depth': 'max', 'depth_range': 'max'}).reset_index()
gdf_tiles_grouped_vert['min_depth'] = gdf_tiles_grouped_vert['min_depth'].round(2)
gdf_tiles_grouped_vert['max_depth'] = gdf_tiles_grouped_vert['max_depth'].round(2)
gdf_tiles_grouped_vert['depth_range'] = gdf_tiles_grouped_vert['depth_range'].round(2)

gdf_tiles_grouped_vert

# Do sorting on different columns to explain differences in the table (analyse!)

# sort on depth_range
# gdf_tiles_grouped_vert.sort_values(by='depth_range', ascending=False, inplace=True)
# gdf_tiles_grouped_vert

,ref_region,min_depth,max_depth,depth_range
0,ARP,-1.26,1.35,2.54
1,ARS,-0.43,-0.21,0.03
2,BOB,-0.46,1.03,1.47
3,CAF,-1.02,0.48,1.49
4,CAR,-0.88,0.57,1.15
5,CAU,-2.48,2.70,5.17
6,CNA,-0.65,0.35,0.90
7,EAO,-1.08,2.18,3.26
8,EAS,-4.08,3.41,6.11
9,EAU,-1.77,2.68,4.20


In [365]:
# to copy into word
# for i in gdf_tiles_grouped_vert["depth_range"]:
#     print(i)

# for reports
gdf_tiles_grouped_vert.depth_range

0      2.54
1      0.03
2      1.47
3      1.49
4      1.15
5      5.17
6      0.90
7      3.26
8      6.11
9      4.20
10     0.80
11     2.30
12     6.77
13     1.46
14     4.31
15     3.81
16     3.60
17     2.83
18     3.52
19     5.73
20     6.02
21     1.58
22     9.17
23     5.64
24     8.78
25     0.94
26     4.52
27     7.03
28     3.17
29     2.45
30     5.91
31    10.22
32     1.76
33     0.21
34     4.93
35     2.84
36     3.75
37     5.57
38     3.56
39     3.69
40     0.27
41     1.25
42     1.20
43     8.56
44     4.01
45     3.30
46     3.46
47     8.22
48     4.17
49     1.58
Name: depth_range, dtype: float64